# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Ahmad Nur Mu'minin  
**NIM:** 202310370311089  
**Username GitHub:** MipowGITHUB  

**Dataset:** `akhyar016/dgph-covid19-social-media` — komentar dan submission Reddit bertema COVID-19.

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** dataset bersih hasil Milestone 2 (`02_data_cleaning.ipynb`) → `data/processed/dgph_bersih.parquet`.
2. **Analisis Temporal / Windowing:** Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, serta DuckDB window function `OVER (...)`.
3. **Visualisasi Interaktif:** 6 visualisasi Plotly.
4. **Perumusan Insight:** 5 temuan berdasarkan hasil visualisasi.

> **Peringatan metodologi.** Data yang dianalisis adalah sampel ±10.000 baris yang diambil dari 100 blok berurutan
> (lihat notebook 01), **bukan sampel acak**. Akibatnya *jumlah* baris per subreddit atau per bulan mencerminkan
> desain sampling, bukan volume percakapan sebenarnya. Karena itu analisis memakai **ukuran yang tidak bergantung pada jumlah
> baris** (median/rata-rata skor, panjang teks, korelasi) dan setiap grafik berbasis hitungan diberi catatan.

## Bagian 1: Inisialisasi dan Memuat Data Bersih

In [1]:
from pathlib import Path

import duckdb
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import polars as pl

pio.renderers.default = "notebook_connected"   # JS dimuat dari CDN agar file notebook kecil
pio.templates.default = "plotly_white"

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "dgph_bersih.parquet"
FIG_DIR = ROOT / "output" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} tidak ada. Jalankan 02_data_cleaning.ipynb terlebih dahulu.")

def simpan(fig, nama):
    """Simpan grafik interaktif sebagai HTML mandiri (di-ignore Git)."""
    fig.write_html(FIG_DIR / f"{nama}.html", include_plotlyjs="cdn")

df = pl.read_parquet(DATA_PATH).with_columns(
    pl.col("text").str.count_matches(r"\S+").alias("jumlah_kata"),
    pl.col("timestamp").dt.hour().alias("jam"),
    pl.col("timestamp").dt.weekday().alias("hari"),     # 1 = Senin ... 7 = Minggu
)
print(f"Data loaded: {df.shape[0]:,} baris x {df.shape[1]} kolom")
print("Periode:", df["timestamp"].min().date(), "s.d.", df["timestamp"].max().date())

Data loaded: 9,753 baris x 18 kolom
Periode: 2020-01-28 s.d. 2022-12-16


### Pemetaan wilayah subreddit

Sebagian besar subreddit mewakili kota/negara bagian/negara (mis. `r/atlanta`, `r/de`), sisanya bersifat tematik
(mis. `r/covidlonghaulers`). Pemetaan koordinat di bawah dibuat manual (titik pusat wilayah) untuk analisis spasial.
`r/coronavirusdownunder` (campuran Australia/Selandia Baru) dan `r/coronavirusfos` tidak jelas lokasinya sehingga
tidak dipetakan.

In [2]:
# suffix source -> (lat, lon, negara)
LOKASI = {
    "anchorage": (61.22, -149.90, "AS"), "atlanta": (33.75, -84.39, "AS"), "austin": (30.27, -97.74, "AS"),
    "baltimore": (39.29, -76.61, "AS"), "brighton": (50.82, -0.14, "Inggris"), "chicago": (41.88, -87.63, "AS"),
    "columbus": (39.96, -83.00, "AS"), "coronabr": (-14.24, -51.93, "Brasil"),
    "coronavirusaustralia": (-25.27, 133.78, "Australia"), "coronaviruscolorado": (39.0, -105.5, "AS"),
    "coronavirusga": (32.7, -83.4, "AS"), "coronavirusmichigan": (44.3, -85.6, "AS"),
    "coronaviruspa": (41.0, -77.8, "AS"), "coronavirusuk": (54.0, -2.0, "Inggris"),
    "coronavirusus": (39.8, -98.6, "AS"), "coronaviruswa": (47.4, -120.7, "AS"),
    "dallas": (32.78, -96.80, "AS"), "de": (51.17, 10.45, "Jerman"), "delaware": (39.0, -75.5, "AS"),
    "denver": (39.74, -104.99, "AS"), "desmoines": (41.59, -93.62, "AS"), "detroit": (42.33, -83.05, "AS"),
    "fargo": (46.88, -96.79, "AS"), "georgia": (32.7, -83.4, "AS"), "germany": (51.17, 10.45, "Jerman"),
    "hawaii": (20.8, -156.3, "AS"), "houston": (29.76, -95.37, "AS"), "indiana": (40.0, -86.1, "AS"),
    "indianapolis": (39.77, -86.16, "AS"), "jerseycity": (40.73, -74.07, "AS"), "kansascity": (39.10, -94.58, "AS"),
    "kentucky": (37.5, -85.3, "AS"), "lockdownskepticismau": (-25.27, 133.78, "Australia"),
    "london": (51.51, -0.13, "Inggris"), "losangeles": (34.05, -118.24, "AS"), "maryland": (39.0, -76.8, "AS"),
    "memphis": (35.15, -90.05, "AS"), "nashville": (36.16, -86.78, "AS"), "neworleans": (29.95, -90.07, "AS"),
    "newhampshire": (43.7, -71.6, "AS"), "northdakota": (47.5, -100.5, "AS"),
    "northernireland": (54.6, -6.9, "Inggris"), "oklahoma": (35.5, -97.5, "AS"),
    "orangecounty": (33.7, -117.8, "AS"), "pennsylvania": (41.0, -77.8, "AS"), "perth": (-31.95, 115.86, "Australia"),
    "philadelphia": (39.95, -75.17, "AS"), "portland": (45.52, -122.68, "AS"), "sacramento": (38.58, -121.49, "AS"),
    "saltlakecity": (40.76, -111.89, "AS"), "sandiego": (32.72, -117.16, "AS"), "sanfrancisco": (37.77, -122.42, "AS"),
    "seattle": (47.61, -122.33, "AS"), "stlouis": (38.63, -90.20, "AS"), "washingtondc": (38.9, -77.04, "AS"),
    "wisconsin": (44.5, -89.5, "AS"),
}
lokasi_df = pl.DataFrame(
    [(f"reddit_{k}", *v) for k, v in LOKASI.items()],
    schema=["source", "lat", "lon", "negara"], orient="row",
)
df = df.join(lokasi_df, on="source", how="left").with_columns(
    pl.col("negara").fill_null("Tematik/Tidak jelas"),
    pl.col("source").str.replace("^reddit_", "").alias("subreddit_nama"),
)
df.group_by("negara").agg(pl.col("source").n_unique().alias("jumlah_subreddit"), pl.len().alias("baris_sampel")).sort("jumlah_subreddit", descending=True)

negara,jumlah_subreddit,baris_sampel
str,u32,u32
"""AS""",46,5073
"""Tematik/Tidak jelas""",10,3013
"""Inggris""",4,580
"""Australia""",3,291
"""Jerman""",2,697
"""Brasil""",1,99


## Bagian 2: Analisis Temporal dan Window Functions (Materi Pertemuan 6)

Tiga operasi: (1) `group_by_dynamic` untuk downsampling ke bulanan, (2) `rolling_mean` + `shift` di Polars,
(3) window function DuckDB `AVG(...) OVER (...)` sebagai pembanding silang.

In [3]:
# 1) Downsampling bulanan
bulanan = (
    df.sort("timestamp")
    .group_by_dynamic("timestamp", every="1mo")
    .agg(
        pl.len().alias("jumlah_baris"),
        pl.col("score").mean().alias("skor_rata2"),
        pl.col("score").median().alias("skor_median"),
        pl.col("panjang_teks").median().alias("panjang_median"),
    )
)

# 2) Rolling mean 3 bulan + selisih bulan-ke-bulan dengan shift
bulanan = bulanan.with_columns(
    pl.col("skor_rata2").rolling_mean(window_size=3).alias("skor_rata2_ma3"),
    pl.col("jumlah_baris").rolling_mean(window_size=3).alias("baris_ma3"),
    (pl.col("jumlah_baris") - pl.col("jumlah_baris").shift(1)).alias("selisih_baris_mom"),
)
bulanan.head(8)

timestamp,jumlah_baris,skor_rata2,skor_median,panjang_median,skor_rata2_ma3,baris_ma3,selisih_baris_mom
"datetime[μs, UTC]",u32,f64,f64,f64,f64,f64,u32
2020-01-01 00:00:00 UTC,17,3.411765,2.0,224.0,null,null,null
2020-02-01 00:00:00 UTC,48,3.083333,2.0,157.0,null,null,31
2020-03-01 00:00:00 UTC,527,5.060721,2.0,149.0,3.85194,197.333333,479
2020-04-01 00:00:00 UTC,767,6.032595,2.0,182.0,4.72555,447.333333,240
2020-05-01 00:00:00 UTC,395,4.194937,2.0,234.0,5.096084,563.0,4294966924
2020-06-01 00:00:00 UTC,602,5.777409,3.0,99.0,5.33498,588.0,207
2020-07-01 00:00:00 UTC,289,11.719723,3.0,170.0,7.23069,428.666667,4294966983
2020-08-01 00:00:00 UTC,214,10.252336,3.0,164.5,9.249823,368.333333,4294967221


In [4]:
# 3) Window function DuckDB sebagai verifikasi silang untuk MA-3 jumlah baris
con = duckdb.connect()
con.register("bulanan", bulanan)
cek = con.sql("""
    SELECT timestamp::DATE AS bulan, jumlah_baris,
           AVG(jumlah_baris) OVER (ORDER BY timestamp ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS ma3_duckdb,
           baris_ma3 AS ma3_polars
    FROM bulanan ORDER BY bulan
""").pl()
# 2 baris pertama tidak lengkap di Polars (null) tetapi parsial di DuckDB, jadi bandingkan mulai baris ke-3
selisih_maks = cek.slice(2).select((pl.col("ma3_duckdb") - pl.col("ma3_polars")).abs().max()).item()
print(f"Selisih maksimum MA-3 Polars vs DuckDB: {selisih_maks:.6f}")
assert selisih_maks < 1e-6
cek.head(6)

Selisih maksimum MA-3 Polars vs DuckDB: 0.000000


bulan,jumlah_baris,ma3_duckdb,ma3_polars
date,u32,f64,f64
2020-01-01,17,17.0,null
2020-02-01,48,32.5,null
2020-03-01,527,197.333333,197.333333
2020-04-01,767,447.333333,447.333333
2020-05-01,395,563.0,563.0
2020-06-01,602,588.0,588.0


## Bagian 3: 6 Visualisasi Interaktif (Plotly)

Seluruh grafik juga disimpan sebagai HTML mandiri di `output/figures/` (di-ignore Git). Pratinjau interaktif
tidak tampil di GitHub; buka notebook di JupyterLab (lihat README) atau buka file HTML.

In [5]:
# Visualisasi 1: Tren waktu interaktif (dengan range slider)
v1 = bulanan
fig1 = go.Figure()
fig1.add_bar(x=v1["timestamp"], y=v1["jumlah_baris"], name="Jumlah baris sampel", marker_color="#cbd5e1", yaxis="y2")
fig1.add_scatter(x=v1["timestamp"], y=v1["skor_rata2"], name="Skor rata-rata", mode="lines+markers", line=dict(color="#94a3b8"))
fig1.add_scatter(x=v1["timestamp"], y=v1["skor_rata2_ma3"], name="Skor rata-rata (MA-3)", mode="lines", line=dict(color="#e11d48", width=3))
fig1.update_layout(
    title="Tren bulanan skor rata-rata (MA-3) vs jumlah baris sampel",
    xaxis=dict(rangeslider=dict(visible=True), title="Bulan"),
    yaxis=dict(title="Skor rata-rata"),
    yaxis2=dict(title="Jumlah baris sampel", overlaying="y", side="right", showgrid=False),
    legend=dict(orientation="h", y=1.12), hovermode="x unified", height=480,
)
simpan(fig1, "01_tren_bulanan"); fig1.show()

In [6]:
# Visualisasi 2: Distribusi skor dan outlier per jenis konten (sumbu log, skor > 0)
pos = df.filter(pl.col("score") > 0)
print(f"Skor <= 0 dikeluarkan dari grafik log: {len(df) - len(pos):,} baris ({1 - len(pos)/len(df):.1%})")
fig2 = px.box(
    pos.to_pandas(), x="content_type", y="score", color="content_type", log_y=True, points="outliers",
    hover_data=["subreddit_nama", "panjang_teks"],
    title="Distribusi skor (log) per jenis konten: ekor panjang dan banyak outlier",
    labels={"score": "Skor (log)", "content_type": "Jenis konten"},
)
fig2.update_layout(showlegend=False, height=480)
simpan(fig2, "02_distribusi_skor"); fig2.show()

Skor <= 0 dikeluarkan dari grafik log: 847 baris (8.7%)


In [7]:
# Visualisasi 3: Korelasi Spearman antar-variabel numerik (rank agar tahan terhadap ekor panjang)
NUM = ["score", "num_comments", "panjang_teks", "jumlah_kata", "jam"]
ranked = df.select([pl.col(c).rank().alias(c) for c in NUM])
corr = ranked.corr()
fig3 = px.imshow(
    corr.to_numpy().round(2), x=NUM, y=NUM, text_auto=True, zmin=-1, zmax=1,
    color_continuous_scale="RdBu_r", title="Korelasi Spearman antar-variabel numerik",
)
fig3.update_layout(height=500)
simpan(fig3, "03_korelasi"); fig3.show()
corr_sub = df.filter(pl.col("content_type") == "reddit_submission").select(
    [pl.col(c).rank().alias(c) for c in ["score", "num_comments", "panjang_teks"]]).corr()
print("Khusus submission (num_comments bermakna di sini):")
corr_sub

Khusus submission (num_comments bermakna di sini):


score,num_comments,panjang_teks
f64,f64,f64
1.0,0.54418,0.137594
0.54418,1.0,-0.012627
0.137594,-0.012627,1.0


In [8]:
# Visualisasi 4: Perilaku segmen, 15 subreddit dengan skor median tertinggi
seg = (
    df.group_by("subreddit_nama", "negara")
    .agg(pl.col("score").median().alias("skor_median"), pl.col("score").mean().round(1).alias("skor_rata2"), pl.len().alias("n"))
    .sort(["skor_median", "skor_rata2"], descending=True).head(15)
)
fig4 = px.bar(
    seg.sort("skor_median").to_pandas(), x="skor_median", y="subreddit_nama", color="negara", orientation="h",
    hover_data=["skor_rata2", "n"], title="15 subreddit dengan skor median tertinggi",
    labels={"skor_median": "Skor median", "subreddit_nama": "Subreddit"},
)
fig4.update_layout(height=520)
simpan(fig4, "04_segmen_subreddit"); fig4.show()

In [9]:
# Visualisasi 5: Pola siklus harian x mingguan (waktu UTC)
fig5 = px.density_heatmap(
    df.to_pandas(), x="jam", y="hari", nbinsx=24, nbinsy=7, color_continuous_scale="Viridis",
    title="Sebaran waktu posting: jam (UTC) x hari dalam minggu (1 = Senin)",
    labels={"jam": "Jam (UTC)", "hari": "Hari (1 = Senin)", "count": "Jumlah baris"},
)
fig5.update_yaxes(dtick=1)
fig5.update_layout(height=480)
simpan(fig5, "05_siklus_jam_hari"); fig5.show()

In [10]:
# Visualisasi 6: Spasial, skor rata-rata per lokasi (ukuran) dan skor median (warna)
geo = (
    df.filter(pl.col("lat").is_not_null())
    .group_by("subreddit_nama", "negara", "lat", "lon")
    .agg(pl.col("score").mean().round(1).alias("skor_rata2"), pl.col("score").median().alias("skor_median"), pl.len().alias("n"))
)
fig6 = px.scatter_geo(
    geo.to_pandas(), lat="lat", lon="lon", size="skor_rata2", color="skor_median", hover_name="subreddit_nama",
    hover_data=["negara", "n"], color_continuous_scale="YlOrRd", size_max=40, projection="natural earth",
    title="Peta subreddit lokal: ukuran = skor rata-rata, warna = skor median",
)
fig6.update_layout(height=560)
simpan(fig6, "06_peta_subreddit"); fig6.show()

## Bukti pendukung insight

In [11]:
# Angka yang dikutip pada Bagian 4
total = df["score"].sum()
k = max(1, int(len(df) * 0.01))
print(f"Skor: median={df['score'].median():.0f}, rata-rata={df['score'].mean():.1f}, maks={df['score'].max()}")
print(f"Top 1% baris ({k}) menyumbang {df.sort('score', descending=True).head(k)['score'].sum() / total:.1%} dari total skor")
print()
print(df.group_by("content_type").agg(
    pl.len().alias("n"), pl.col("score").median().alias("skor_median"), pl.col("score").mean().round(1).alias("skor_rata2"),
    pl.col("panjang_teks").median().alias("panjang_median"), pl.col("num_comments").mean().round(2).alias("komentar_rata2"),
))
print()
bin_len = (
    df.with_columns(
        pl.when(pl.col("panjang_teks") <= 50).then(pl.lit("1: <=50"))
        .when(pl.col("panjang_teks") <= 150).then(pl.lit("2: 51-150"))
        .when(pl.col("panjang_teks") <= 400).then(pl.lit("3: 151-400"))
        .when(pl.col("panjang_teks") <= 800).then(pl.lit("4: 401-800"))
        .otherwise(pl.lit("5: >800")).alias("bin_panjang"))
    .group_by("bin_panjang").agg(pl.len().alias("n"), pl.col("score").median().alias("skor_median"), pl.col("score").mean().round(1).alias("skor_rata2"))
    .sort("bin_panjang")
)
print(bin_len)
print()
print(bulanan.sort("jumlah_baris", descending=True).head(3).select("timestamp", "jumlah_baris", "skor_rata2"))
print(bulanan.sort("skor_rata2", descending=True).head(3).select("timestamp", "jumlah_baris", "skor_rata2"))
print()
pl.Config.set_tbl_rows(20)
print(df.group_by("negara").agg(pl.len().alias("n"), pl.col("score").median().alias("skor_median"), pl.col("score").mean().round(1).alias("skor_rata2"),
                                 pl.col("panjang_teks").median().alias("panjang_median")).sort("n", descending=True))

Skor: median=2, rata-rata=6.6, maks=1661.0
Top 1% baris (97) menyumbang 25.3% dari total skor

shape: (2, 6)
┌───────────────────┬──────┬─────────────┬────────────┬────────────────┬────────────────┐
│ content_type      ┆ n    ┆ skor_median ┆ skor_rata2 ┆ panjang_median ┆ komentar_rata2 │
│ ---               ┆ ---  ┆ ---         ┆ ---        ┆ ---            ┆ ---            │
│ str               ┆ u32  ┆ f64         ┆ f64        ┆ f64            ┆ f64            │
╞═══════════════════╪══════╪═════════════╪════════════╪════════════════╪════════════════╡
│ reddit_submission ┆ 389  ┆ 6.0         ┆ 31.6       ┆ 687.0          ┆ 7.62           │
│ reddit_comment    ┆ 9364 ┆ 2.0         ┆ 5.6        ┆ 149.0          ┆ 0.62           │
└───────────────────┴──────┴─────────────┴────────────┴────────────────┴────────────────┘

shape: (5, 4)
┌─────────────┬──────┬─────────────┬────────────┐
│ bin_panjang ┆ n    ┆ skor_median ┆ skor_rata2 │
│ ---         ┆ ---  ┆ ---         ┆ ---        │
│ str 

## Bagian 4: Rumusan 5 Insight Analitik Utama

> Semua angka berasal dari sampel 9.753 baris bersih (lihat peringatan metodologi di awal). Insight sengaja dirumuskan pada
> ukuran yang tidak bergantung pada jumlah baris per grup.

1. **Insight 1 (Tren Waktu & Musiman): rata-rata skor bulanan didorong segelintir konten viral, bukan pergeseran keterlibatan umum.**
   Pada bulan dengan ≥100 baris, skor *rata-rata* berayun dari 1,5 (Sep 2022) sampai 16,5 (Mar 2021), sedangkan skor *median* bulanan
   berada di 2–3 pada 29 dari 36 bulan. Bulan dengan rata-rata tertinggi (Mar 2021, Nov 2020, Okt 2022) justru bukan bulan dengan baris
   terbanyak; lonjakannya berasal dari beberapa postingan bernilai tinggi. Tren MA-3 (Visualisasi 1) karenanya harus dibaca bersama median.
   Puncak jumlah baris pada Mar–Apr 2020 hanya mencerminkan letak blok sampel dan tidak dipakai sebagai bukti lonjakan diskusi.
   Pola jam×hari (Visualisasi 5) juga hanya indikatif karena waktu UTC mencampur zona waktu AS, Eropa, dan Australia.

2. **Insight 2 (Distribusi & Anomali): skor sangat timpang (ekor panjang).**
   Median skor hanya 2 dan rata-rata 6,6, tetapi maksimum 1.661. Hanya 1% baris teratas (97 baris) menyumbang 25,3% dari total skor, dan
   8,7% baris bernilai ≤ 0 (tidak tampil di grafik log). Untuk ringkasan statistik atau model, median, log-skor, atau winsorization
   (`score_clip` dari Milestone 2) lebih tepat daripada rata-rata mentah.

3. **Insight 3 (Korelasi Kunci): skor berkaitan kuat dengan jumlah komentar, tetapi lemah dengan panjang teks.**
   Pada submission, korelasi Spearman skor–`num_comments` = 0,54: konten yang disukai juga dibahas. Korelasi skor–panjang teks hanya sekitar 0,14.
   Rata-rata skor naik dari 3,8 (≤50 karakter) ke 15,5 (>800 karakter), tetapi median hanya naik dari 2 ke 3,5, jadi efek panjang teks
   sebagian besar adalah efek ekor (postingan panjang lebih sering viral), bukan peningkatan skor yang merata.
   `num_comments` hanya bermakna untuk submission (rata-rata 0,62 pada komentar vs 7,62 pada submission).

4. **Insight 4 (Perilaku Kategori / Segmen): submission jauh lebih "berbobot" daripada komentar; antar-komunitas ada selisih, tetapi perlu hati-hati.**
   Submission (4,0% baris) memiliki skor median 6 vs 2 untuk komentar, skor rata-rata 31,6 vs 5,6, dan teks median 687 vs 149 karakter.
   Antar-subreddit, median tertinggi adalah `coronaviruswa`, `lockdownskepticism`, dan `pennsylvania` (masing-masing 6, sekitar 3× median keseluruhan).
   Setiap subreddit hanya diwakili sekitar satu blok ±100 baris (satu thread), sehingga selisih ini bisa berasal dari thread tertentu
   dan bukan sifat komunitas. Peta (Visualisasi 6) menunjukkan 46 dari 56 subreddit yang terpetakan berasal dari AS, jadi dataset condong ke
   diskusi lokal Amerika.

5. **Insight 5 (Rekomendasi Praktis): rancang analisis data penuh dengan sampel acak dan metrik robust.**
   (a) Pakai median/log-skor dan sertakan interval, bukan rata-rata, untuk membandingkan bulan atau komunitas.
   (b) Analisis lanjutan (topik/sentimen, NLP) sebaiknya dipisahkan antara submission dan komentar karena karakteristiknya berbeda jauh.
   (c) Untuk menarik kesimpulan volume atau tren sebenarnya, ambil sampel acak per shard (DuckDB `USING SAMPLE` atau `scan_parquet` dengan
   filter) alih-alih blok berurutan, dan bandingkan hasilnya dengan sampel ini.
   (d) Buang `[removed]`/`[deleted]` sejak awal (2,4% dari sampel) agar tidak mengganggu analisis teks.